# Phase 2C — WLPA & the Playback Singers
Requires Phase 1 (`processed/`) and Phase 2A checkpoints/predictions (`results2A/`).

| # | Experiment | Question |
|---|---|---|
| C1 | **WLPA via restricted softmax** (free, from saved R3 logits) | Within each same-state, same-language pair, can the artist-disjoint 15-way model tell the two styles apart? |
| C2 | **WLPA via dedicated binary models** + MFCC+RF pair baseline | The clean measurement: binary, artist-disjoint, song-aggregated, chance = 50%. |
| C3 | **The four playback singers** (Paudwal, Garg, Kalyanpur, Mangeshkar) | On their *minority*-genre recordings, does the R2 model follow the **music** or the **voice**? |

The pairs (language & region held constant ⇒ language shortcuts impossible):
**Bauls ↔ Bhatiali** (West Bengal, Bengali — the hard, honest test) ·
**Kajri ↔ Sohar** (UP, Bhojpuri) ·
**Bhavageethe ↔ Veeragase** (Karnataka, Kannada — voice vs percussion, positive control).

All outputs → `results2C/` (smoke runs isolated in `results2C_smoke/`). C2 is resumable per run.


In [ ]:
# ============ CELL 0: switches ============
import os
os.environ['IKS_SMOKE']    = '0'   # '1' = tiny pipeline check (results2C_smoke); '0' = full run
os.environ['IKS_RESET_2C'] = '0'   # set '1' ONCE to wipe stale results2C, then back to '0'

In [ ]:
# ============ CELL 1: Setup ============
import os, gc, glob, json, time, shutil, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

if os.path.basename(os.getcwd()) == 'notebooks': os.chdir('..')   # resolve paths from the repo root
PROC_ROOT  = os.environ.get('IKS_PROC_ROOT', './processed')
RES2A      = os.environ.get('IKS_RES2A', f"{PROC_ROOT}/results2A")
SMOKE = os.environ.get('IKS_SMOKE', '0') == '1'
RES2C = os.environ.get('IKS_RES2C', f"{PROC_ROOT}/results2C")
RES2C = RES2C + "_smoke" if SMOKE else RES2C
os.makedirs(RES2C, exist_ok=True)
if os.environ.get('IKS_RESET_2C', '0') == '1':
    shutil.rmtree(RES2C, ignore_errors=True); os.makedirs(RES2C, exist_ok=True); print(f"RES2C wiped: {RES2C}")
print(('*'*70 + '\n*  SMOKE MODE — tiny truncated data, results are NOT valid  *\n' + '*'*70)
      if SMOKE else f"FULL MODE — results -> {RES2C}")
assert glob.glob(f"{RES2A}/R2_resnet18_fold*_preds.npz"), "2A prediction files not found — run Phase 2A first"

idxdf = pd.read_parquet(f"{PROC_ROOT}/index_with_folds.parquet").sort_values('global_idx').reset_index(drop=True)
assert (idxdf['global_idx'].values == np.arange(len(idxdf))).all()
idxdf = idxdf.set_index('global_idx', drop=False)
for c in ['genre_id','artist_id','state_id','gender_id','fold_artist','fold_song']:
    idxdf[c] = idxdf[c].astype(int)
N_TOTAL = len(idxdf); N_CLASSES = idxdf['genre_id'].nunique()
ID2GENRE = idxdf.drop_duplicates('genre_id').set_index('genre_id')['genre'].to_dict()
GENRE2ID = {v: k for k, v in ID2GENRE.items()}

blocks = idxdf.groupby('style')['global_idx'].agg(['min','max','count']).sort_values('min')
X_ALL = np.empty((N_TOTAL, 128, 130), dtype=np.float16)
t0 = time.time()
for s, r in blocks.iterrows():
    Xs = np.load(f"{PROC_ROOT}/X_{s}.npy", mmap_mode='r'); X_ALL[r['min']:r['max']+1] = Xs[:]; del Xs
print(f"X_ALL in RAM: {X_ALL.nbytes/1e9:.2f} GB ({time.time()-t0:.0f}s)")

STRIDE = 6
idxdf['seg_rank'] = idxdf.groupby('source').cumcount()
stride_pool = idxdf.loc[idxdf['seg_rank'] % STRIDE == 0, 'global_idx'].values
GENRE_ARR = idxdf['genre_id'].values

# artist display names live in meta_all.parquet (Phase 1); fall back to ids if unavailable
if 'artist' not in idxdf.columns:
    try:
        nm = pd.read_parquet(f"{PROC_ROOT}/meta_all.parquet")[['source', 'artist']].drop_duplicates('source')
        idxdf = idxdf.merge(nm, on='source', how='left').set_index('global_idx', drop=False)
        idxdf['artist'] = idxdf['artist'].fillna('id' + idxdf['artist_id'].astype(str))
    except Exception:
        idxdf['artist'] = 'id' + idxdf['artist_id'].astype(str)

DEFAULT_PAIRS = [("Bauls", "Bhatiali"), ("Kajri", "Sohar"), ("Bhavageethe", "Veeragase")]
PAIRS = json.loads(os.environ.get('IKS_PAIRS_JSON', 'null')) or DEFAULT_PAIRS
PAIRS = [(a, b) for a, b in PAIRS if a in GENRE2ID and b in GENRE2ID]
print("pairs:", PAIRS, f"| SMOKE={SMOKE}")

import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else '(CPU)')

In [ ]:
# ============ CELL 2: Models & utilities (verbatim from 2A/2B) ============
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score, accuracy_score

def normalize_batch(x): return (x + 40.0) / 40.0

class BasicBlock(nn.Module):
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.c1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False); self.b1 = nn.BatchNorm2d(cout)
        self.c2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False);     self.b2 = nn.BatchNorm2d(cout)
        self.down = None
        if stride != 1 or cin != cout:
            self.down = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))
    def forward(self, x):
        idn = x if self.down is None else self.down(x)
        h = F.relu(self.b1(self.c1(x))); h = self.b2(self.c2(h))
        return F.relu(h + idn)

class ResNet18(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(1, 64, 7, 2, 3, bias=False), nn.BatchNorm2d(64),
                                  nn.ReLU(inplace=True), nn.MaxPool2d(3, 2, 1))
        def layer(cin, cout, s): return nn.Sequential(BasicBlock(cin, cout, s), BasicBlock(cout, cout))
        self.l1 = layer(64, 64, 1);  self.l2 = layer(64, 128, 2)
        self.l3 = layer(128, 256, 2); self.l4 = layer(256, 512, 2)
        self.fc = nn.Linear(512, n_classes)
    def forward(self, x):
        h = self.stem(x); h = self.l4(self.l3(self.l2(self.l1(h))))
        return self.fc(F.adaptive_avg_pool2d(h, 1).flatten(1))

def make_model(name, n_classes): return {'resnet18': ResNet18}[name](n_classes)

def metric_pack(y, p, prefix):
    labels = np.unique(y)
    return {f'{prefix}_acc': float(accuracy_score(y, p)),
            f'{prefix}_macro_f1': float(f1_score(y, p, labels=labels, average='macro', zero_division=0))}

def song_agg(gidx, logits, label_arr):
    src = idxdf.loc[gidx, 'source'].values
    L = pd.DataFrame(logits); L['source'] = src
    agg = L.groupby('source').mean()
    ytrue = pd.Series(label_arr[gidx], index=src).groupby(level=0).first().loc[agg.index].values
    return ytrue, agg.values.argmax(1), agg.index.values

@torch.no_grad()
def predict(model, gidx, batch=1024):
    model.eval(); outs = []
    for i in range(0, len(gidx), batch):
        xb = normalize_batch(torch.from_numpy(X_ALL[gidx[i:i+batch]].astype(np.float32)).unsqueeze(1)).to(DEVICE)
        with torch.autocast(DEVICE.type, enabled=(DEVICE.type == 'cuda')):
            outs.append(model(xb).float().cpu())
    return torch.cat(outs).numpy()

def train_generic(model_name, tr, va, te, label_arr, n_classes, epochs, batch=512, lr=3e-4, seed=0, log_every=10):
    torch.manual_seed(seed); np.random.seed(seed)
    model = make_model(model_name, n_classes).to(DEVICE)
    counts = np.bincount(label_arr[tr], minlength=n_classes).astype(np.float64)
    w = np.where(counts > 0, counts.sum() / (n_classes * np.maximum(counts, 1)), 0.0)
    crit  = nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32, device=DEVICE))
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = torch.amp.GradScaler(enabled=(DEVICE.type == 'cuda'))
    rng = np.random.default_rng(seed)
    best_f1, best_state = -1.0, None
    t0 = time.time()
    for ep in range(epochs):
        model.train(); perm = rng.permutation(tr)
        for i in range(0, len(perm), batch):
            gb = perm[i:i+batch]
            xb = normalize_batch(torch.from_numpy(X_ALL[gb].astype(np.float32)).unsqueeze(1)).to(DEVICE)
            yb = torch.from_numpy(label_arr[gb]).to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(DEVICE.type, enabled=(DEVICE.type == 'cuda')):
                loss = crit(model(xb), yb)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        sched.step()
        pv = predict(model, va).argmax(1); yv = label_arr[va]
        f1v = f1_score(yv, pv, labels=np.unique(yv), average='macro', zero_division=0)
        if f1v > best_f1:
            best_f1 = float(f1v)
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        if ep % log_every == 0 or ep == epochs - 1:
            print(f"    ep {ep:>3}  val macroF1={f1v:.4f} ({time.time()-t0:.0f}s)", flush=True)
    model.load_state_dict(best_state)
    return model, predict(model, te), best_f1

def carve_val_by_artist(tr, fold, frac=0.10):
    rng = np.random.default_rng(1000 + fold)
    g = idxdf.loc[tr, 'artist_id'].values
    ug = rng.permutation(np.unique(g)); nv = max(1, int(round(len(ug) * frac)))
    vmask = np.isin(g, ug[:nv]); return tr[~vmask], tr[vmask]
print('cell 2 ready')

In [ ]:
# ============ CELL 3 (C1): WLPA via restricted softmax from saved R3 logits (free) ============
# For each pair, take the artist-disjoint R3 test predictions, keep only recordings of the two
# genres, and decide between JUST those two logits (mean over the recording's segments).
rows = []
for mname in ['resnet18', 'crnn']:
    for p in sorted(glob.glob(f"{RES2A}/R3_{mname}_fold*_preds.npz")):
        fold = int(p.split('fold')[1].split('_')[0])
        z = np.load(p, allow_pickle=True)
        gidx, logits = z['gidx'], z['logits']
        for (A, B) in PAIRS:
            ia, ib = GENRE2ID[A], GENRE2ID[B]
            m = np.isin(GENRE_ARR[gidx], [ia, ib])
            if m.sum() == 0: continue
            g2, l2 = gidx[m], logits[m][:, [ia, ib]]
            lab = (GENRE_ARR == ib).astype(np.int64)          # 0=A, 1=B
            ys, ps, _ = song_agg(g2, l2, lab)
            if len(np.unique(ys)) < 2:
                continue                                       # fold lacks one side of the pair
            rows.append(dict(method='R3-restricted', model=mname, pair=f"{A}|{B}", fold=fold,
                             n_songs=len(ys), song_acc=float((ys == ps).mean())))
C1 = pd.DataFrame(rows)
if len(C1):
    s1 = C1.groupby(['pair', 'model']).agg(song_acc=('song_acc','mean'), std=('song_acc','std'),
                                           folds=('fold','nunique'), songs=('n_songs','sum'))
    print('================ C1: WLPA — restricted softmax (chance = 0.50) ================')
    print(s1.round(4).to_string())
    C1.to_csv(f"{RES2C}/wlpa_restricted.csv", index=False)
else:
    print('no valid pair/fold combinations found in R3 predictions')

In [ ]:
# ============ CELL 4 (C2): WLPA — dedicated binary models + MFCC+RF baseline (resumable) ============
from scipy.fftpack import dct
from sklearn.ensemble import RandomForestClassifier

def mfcc_feats(gidx, batch=8192):
    out = np.empty((len(gidx), 78), np.float32)
    for i in range(0, len(gidx), batch):
        x = X_ALL[gidx[i:i+batch]].astype(np.float32)
        M  = dct(x, type=2, axis=1, norm='ortho')[:, :13, :]
        d1 = np.diff(M, axis=2); d2 = np.diff(d1, axis=2)
        out[i:i+batch] = np.concatenate([M.mean(2), M.std(2), d1.mean(2), d1.std(2),
                                         d2.mean(2), d2.std(2)], axis=1)
    return out

EPOCHS_BIN = 2 if SMOKE else 25
for (A, B) in PAIRS:
    ia, ib = GENRE2ID[A], GENRE2ID[B]
    lab = (GENRE_ARR == ib).astype(np.int64)
    pool = stride_pool[np.isin(GENRE_ARR[stride_pool], [ia, ib])]
    for fold in range(5):
        jp = f"{RES2C}/wlpa_bin_{A}_{B}_fold{fold}.json".replace(' ', '_')
        if os.path.exists(jp): print(f"[skip] {A}|{B} fold{fold}"); continue
        f = idxdf.loc[pool, 'fold_artist'].values
        te, tr = pool[f == fold], pool[f != fold]
        if SMOKE: tr, te = tr[:1500], te[:600]
        if len(te) == 0 or len(np.unique(lab[te])) < 2 or len(np.unique(lab[tr])) < 2:
            json.dump(dict(pair=f"{A}|{B}", fold=fold, skipped='missing class'), open(jp, 'w'))
            print(f"[degenerate] {A}|{B} fold{fold} — one side absent, skipped"); continue
        tr2, va = carve_val_by_artist(tr, fold)
        if len(np.unique(lab[va])) < 2:                       # val drew a single class -> stratified fallback
            rng = np.random.default_rng(fold); va_parts = []
            for cval in np.unique(lab[tr]):
                idc = rng.permutation(tr[lab[tr] == cval])
                va_parts.append(idc[:max(1, len(idc)//10)])
            va = np.concatenate(va_parts); tr2 = tr[~np.isin(tr, va)]
        print(f"[{A}|{B} fold{fold}] train={len(tr2):,} test={len(te):,}", flush=True)
        _, logits, _ = train_generic('resnet18', tr2, va, te, lab, 2, epochs=EPOCHS_BIN, seed=fold)
        ys, ps, _ = song_agg(te, logits, lab)
        m = metric_pack(lab[te], logits.argmax(1), 'seg'); m.update(metric_pack(ys, ps, 'song'))
        # MFCC+RF on the identical split
        rf = RandomForestClassifier(n_estimators=60 if SMOKE else 300, n_jobs=-1,
                                    class_weight='balanced_subsample', random_state=fold)
        rf.fit(mfcc_feats(tr2), lab[tr2])
        proba = rf.predict_proba(mfcc_feats(te))
        ys2, ps2, _ = song_agg(te, proba, lab)
        m.update({'rf_seg_acc': float((proba.argmax(1) == lab[te]).mean()),
                  'rf_song_acc': float((ys2 == ps2).mean()),
                  'pair': f"{A}|{B}", 'fold': fold, 'n_songs': int(len(ys))})
        json.dump(m, open(jp, 'w'))
        print(f"    binary song acc={m['song_acc']:.4f} | RF song acc={m['rf_song_acc']:.4f} "
              f"({m['n_songs']} songs)", flush=True)
        gc.collect()
        if DEVICE.type == 'cuda': torch.cuda.empty_cache()

res = [json.load(open(p)) for p in sorted(glob.glob(f"{RES2C}/wlpa_bin_*fold*.json"))]
C2 = pd.DataFrame([r for r in res if 'skipped' not in r])
if len(C2):
    s2 = C2.groupby('pair').agg(bin_song=('song_acc','mean'), bin_std=('song_acc','std'),
                                rf_song=('rf_song_acc','mean'), folds=('fold','nunique'),
                                songs=('n_songs','sum'))
    print('\n================ C2: WLPA — dedicated binary + MFCC+RF (chance = 0.50) ================')
    print(s2.round(4).to_string())
    C2.to_csv(f"{RES2C}/wlpa_binary.csv", index=False)

In [ ]:
# ============ CELL 5 (C3): The four playback singers ============
# Every recording appears exactly once as R2 test across the 5 song-folds. Collect the song-level
# R2 predictions for all recordings, then look at the cross-style artists.
pred_map = {}
for p in sorted(glob.glob(f"{RES2A}/R2_resnet18_fold*_preds.npz")):
    z = np.load(p, allow_pickle=True)
    ys, ps, srcs = song_agg(z['gidx'], z['logits'], GENRE_ARR)
    for s, yt, yp in zip(srcs, ys, ps): pred_map[s] = (int(yt), int(yp))

rec = idxdf.drop_duplicates('source')[['source','song','genre','genre_id','artist','artist_id']].copy()
span = rec.groupby('artist_id')['genre'].nunique()
cross_ids = span[span > 1].index.tolist()
print(f"cross-style artists: {len(cross_ids)}")
if len(cross_ids) == 0:
    print('none found — nothing to do here')
else:
    cs = rec[rec.artist_id.isin(cross_ids)].copy()
    cs['pred_genre'] = cs['source'].map(lambda s: ID2GENRE.get(pred_map.get(s, (None, None))[1], None))
    cs = cs.dropna(subset=['pred_genre'])
    maj = cs.groupby('artist_id')['genre'].agg(lambda g: g.mode().iloc[0]).to_dict()
    cs['artist_majority'] = cs['artist_id'].map(maj)
    cs['rec_type'] = np.where(cs['genre'] == cs['artist_majority'], 'majority-genre', 'minority-genre')
    cs['outcome'] = np.select(
        [cs['pred_genre'] == cs['genre'],
         cs['pred_genre'] == cs['artist_majority']],
        ['correct', 'voice-followed (pred = singer majority)'], default='other error')

    print('\n================ per-recording table ================')
    print(cs[['artist','song','genre','pred_genre','rec_type','outcome']]
          .sort_values(['artist','rec_type']).to_string(index=False))
    summ = (cs.groupby(['artist','rec_type'])['outcome']
              .value_counts(normalize=True).unstack(fill_value=0).round(3))
    print('\n================ summary (proportions) ================')
    print(summ.to_string())
    cs.to_csv(f"{RES2C}/playback_singers.csv", index=False)

    # headline number: on minority-genre recordings, how often does the prediction follow the VOICE?
    mino = cs[cs.rec_type == 'minority-genre']
    if len(mino):
        vf = (mino['outcome'] == 'voice-followed (pred = singer majority)').mean()
        cr = (mino['outcome'] == 'correct').mean()
        print(f"\nMINORITY-GENRE recordings (n={len(mino)}): correct={cr:.1%}  voice-followed={vf:.1%}")

    # figure: stacked outcome bars per artist x recording-type
    grp = (cs.groupby(['artist','rec_type'])['outcome']
             .value_counts(normalize=True).unstack(fill_value=0))
    order_cols = [c for c in ['correct','voice-followed (pred = singer majority)','other error'] if c in grp.columns]
    ax = grp[order_cols].plot(kind='bar', stacked=True, figsize=(10, 4.5),
                              color=['#2ca02c', '#d62728', '#7f7f7f'])
    ax.set_ylabel('proportion of recordings'); ax.set_ylim(0, 1)
    ax.set_title('R2 predictions for the four playback singers — music or voice?')
    plt.xticks(rotation=30, ha='right'); plt.legend(fontsize=8); plt.tight_layout()
    plt.savefig(f"{RES2C}/fig_playback_singers.png", dpi=150); plt.show()

In [ ]:
# ============ CELL 6: Combined WLPA figure ============
frames = []
try:
    c1 = pd.read_csv(f"{RES2C}/wlpa_restricted.csv")
    c1 = c1[c1.model == 'resnet18'].groupby('pair')['song_acc'].agg(['mean','std']).assign(method='R3 restricted softmax')
    frames.append(c1)
except Exception: pass
try:
    c2 = pd.read_csv(f"{RES2C}/wlpa_binary.csv")
    frames.append(c2.groupby('pair')['song_acc'].agg(['mean','std']).assign(method='dedicated binary'))
    frames.append(c2.groupby('pair')['rf_song_acc'].agg(['mean','std']).assign(method='MFCC + RF'))
except Exception: pass

if frames:
    W = pd.concat(frames).reset_index()
    pairs_order = sorted(W['pair'].unique())
    methods = W['method'].unique()
    x = np.arange(len(pairs_order)); width = 0.8 / max(len(methods), 1)
    plt.figure(figsize=(9, 5))
    for k, meth in enumerate(methods):
        sub = W[W.method == meth].set_index('pair').reindex(pairs_order)
        plt.bar(x + k*width, sub['mean'], width, yerr=np.nan_to_num(sub['std']), capsize=3, label=meth)
    plt.axhline(0.5, ls='--', c='k', lw=1); plt.text(-0.35, 0.512, 'chance', fontsize=8)
    plt.xticks(x + width*(len(methods)-1)/2, pairs_order, rotation=15)
    plt.ylabel('song-level accuracy'); plt.ylim(0, 1.02)
    plt.title('WLPA — within-language pair accuracy (artist-disjoint, song-aggregated)')
    plt.legend(); plt.grid(alpha=.3, axis='y'); plt.tight_layout()
    plt.savefig(f"{RES2C}/fig_wlpa.png", dpi=150); plt.show()
    W.to_csv(f"{RES2C}/wlpa_all_methods.csv", index=False)
print('All 2C artifacts saved to', RES2C)

## After running, send me
1. **C1 table** (restricted-softmax WLPA).
2. **C2 table** (dedicated binary + MFCC+RF WLPA) — the paper's WLPA numbers.
3. **C3**: the per-recording table + summary proportions + the `MINORITY-GENRE ... voice-followed=` line,
   and `fig_playback_singers.png`.
4. `fig_wlpa.png`.

This closes the experimental record. Next deliverable from me: the full LaTeX draft (§1–§7) with every
table/figure slotted in.
